# [Baseline] RandomForest — 풍력발전량 예측

기상 예보 데이터(LDAPS/GFS)로 KPX 발전그룹 3곳의 시간대별 **풍력발전량(kWh)** 을 예측하는 회귀 베이스라인입니다.

- **입력**: 예측 시각별 LDAPS/GFS 기상 예보 격자 평균값 + 시간·요일·계절성 캘린더 변수
- **출력**: `kpx_group_1/2/3` 세 발전그룹의 시간대별 예측 발전량(kWh)
- **모델**: 그룹별 `RandomForestRegressor` (Label 제공 기간이 그룹마다 다르므로 각각 따로 학습)

이 대회는 예측 결과 CSV(`baseline_submit.csv`)를 제출하는 방식이라, 데이터 로드부터 제출 파일 생성까지
이 노트북 하나로 처리합니다.


## 1. 라이브러리 불러오기

데이터 처리(pandas, numpy)와 회귀 모델 학습(scikit-learn)에 필요한 라이브러리를 불러옵니다.


In [20]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer


## 2. 데이터 로드

발전량 Label(`train_labels.csv`), 제출 양식(`sample_submission.csv`), LDAPS/GFS 기상 예보 데이터를 불러오고,
날짜 컬럼을 datetime으로 변환합니다. `CAPACITY_KWH` 는 그룹별 설비 용량으로, 이후 예측값을 클리핑하는 데
사용합니다.


In [21]:
DATA_DIR = Path(r"C:\Users\user\iCloudDrive\riversun\동국대학교\3학년 1학기\대외활동\DACON\데이터\open")
TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"

TARGET_COLS = ["kpx_group_1", "kpx_group_2", "kpx_group_3"]
CAPACITY_KWH = {
    "kpx_group_1": 21600,
    "kpx_group_2": 21600,
    "kpx_group_3": 21000,
}

train_labels = pd.read_csv(TRAIN_DIR / "train_labels.csv", encoding="utf-8-sig")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv", encoding="utf-8-sig")

ldaps_train = pd.read_csv(TRAIN_DIR / "ldaps_train.csv", encoding="utf-8-sig")
gfs_train = pd.read_csv(TRAIN_DIR / "gfs_train.csv", encoding="utf-8-sig")
ldaps_test = pd.read_csv(TEST_DIR / "ldaps_test.csv", encoding="utf-8-sig")
gfs_test = pd.read_csv(TEST_DIR / "gfs_test.csv", encoding="utf-8-sig")

train_labels["kst_dtm"] = pd.to_datetime(train_labels["kst_dtm"])
sample_submission["forecast_kst_dtm"] = pd.to_datetime(sample_submission["forecast_kst_dtm"])

scada_vestas = pd.read_csv(TRAIN_DIR / "scada_vestas_train.csv", encoding="utf-8-sig")
scada_unison = pd.read_csv(TRAIN_DIR / "scada_unison_train.csv", encoding="utf-8-sig")

print("train_labels:", train_labels.shape)
print("sample_submission:", sample_submission.shape)
print("ldaps_train:", ldaps_train.shape, "gfs_train:", gfs_train.shape)
print("ldaps_test:", ldaps_test.shape, "gfs_test:", gfs_test.shape)


train_labels: (26304, 4)
sample_submission: (8760, 5)
ldaps_train: (420864, 35) gfs_train: (236736, 40)
ldaps_test: (140160, 35) gfs_test: (78840, 40)


## 3. 데이터 전처리
P1. SCADA 글리치 제거

P2. Group 3 라벨 결측 처리

P3. NWP U/V → 풍속/풍향 변환

P4. Grid 선택

P5. NWP 풍속 Bias Correction

P6. Downtime 구간 플래깅

P7. Temporal Feature Engineering

P8. data_available_kst_dtm 활용 (leakage 방지)

P9. 10% 제외 규칙을 학습에 반영

P10. Train/Validation Split

In [22]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

# 시간 컬럼 datetime 변환 (P10)
for df in [train_labels, scada_vestas, scada_unison, ldaps_train, gfs_train]:
    # 각 파일의 시간 컬럼명에 맞게 변환 (예: kst_dtm, forecast_kst_dtm 등)
    time_col = [col for col in df.columns if 'dtm' in col or 'time' in col][0]
    df[time_col] = pd.to_datetime(df[time_col])

# ==========================================
# P8 & 2대회 함정방지: NWP 중복 제거 및 가용성 필터링
# ==========================================
def clean_nwp(df):
    # data_available_kst_dtm이 예측대상시점(kst_dtm) 이전인 것만 필터링 (Leakage 방지)
    # ※ 컬럼명이 다를 수 있으니 확인 필요
    df = df[df['data_available_kst_dtm'] <= df['forecast_kst_dtm']]
    
    # 동일한 예측대상시점 중 '가장 최근에 발표된 예보'만 남기고 중복 제거
    df = df.sort_values(by=['forecast_kst_dtm', 'data_available_kst_dtm'])
    df = df.drop_duplicates(subset=['forecast_kst_dtm', 'grid_id'], keep='last')
    return df

print("Cleaning NWP data...")
ldaps_train = clean_nwp(ldaps_train)
gfs_train = clean_nwp(gfs_train)

# ==========================================
# P3 & P4: NWP 핵심 격자 선택 및 풍속/풍향 변환
# ==========================================
def process_nwp_features(df, prefix='ldaps'):
    # prefix에 따라 완벽하게 검증된 실제 컬럼명을 강제로 고정(하드코딩)합니다.
    if prefix == 'ldaps':
        u_col = 'heightAboveGround_10_10u'
        v_col = 'heightAboveGround_10_10v'
        t_col = 'heightAboveGround_2_t'       # 지상 2m 기온 (K)
        p_col = 'surface_0_sp'                # 지표면 기압 (Pa)
        h_col = 'heightAboveGround_2_r'       # 상대습도 (%)
    else:  # gfs인 경우
        u_col = 'heightAboveGround_10_10u'
        v_col = 'heightAboveGround_10_10v'
        t_col = 'heightAboveGround_2_2t'      # GFS 실제 지상 2m 기온 (K)
        p_col = 'surface_0_sp'                # GFS 지표면 기압 (Pa)
        h_col = 'heightAboveGround_2_2r'      # GFS 지상 2m 상대습도 (%)
    
    # ----------------------------------------------------
    # [물리 단위 영점 조절] LDAPS / GFS 공통 변환
    # ----------------------------------------------------
    # 1. 기온 보정: 켈빈(K) -> 섭씨(°C) 변환
    temp_c = df[t_col] - 273.15
    temp_k = df[t_col] # 켈빈 기온 그대로 수식 분모에 주입
    
    # 2. 기압 보정: 파스칼(Pa) -> 헥토파스칼(hPa) 변환 (두 모델 모두 10만 단위 확인 완료)
    press_hpa = df[p_col] / 100.0
    
    # ① 대기밀도 (air_density) 계산 (1위 마스터 물리 수식)
    e_s = 6.1078 * 10 ** ((7.5 * temp_c) / (temp_c + 237.3))
    e = (df[h_col] / 100.0) * e_s
    p_dry = press_hpa - e
    df['air_density'] = (p_dry / (287.058 * temp_k)) + ((e / (461.495 * temp_k)))
    
    # ② 절대습도 (absolute_humidity) 계산
    alpha = ((17.27 * temp_c) / (237.7 + temp_c)) + np.log(0.5)
    dew_point = (237.7 * alpha) / (17.27 - alpha)
    e_s_dew = 6.112 * np.exp((17.67 * dew_point) / (dew_point + 243.5))
    e_a = e_s_dew * (df[h_col] / 100.0)
    e_a_Pa = e_a * 100.0
    rho_d = (press_hpa * 100.0) / (287.058 * temp_k)
    rho_v = (e_a_Pa / (461.495 * temp_k))
    df['absolute_humidity'] = (rho_v / (rho_d + rho_v)) * 1000.0
    
    # ③ 공기분압 (air_pressure) 계산
    e_s_air = 6.112 * np.exp((17.67 * temp_c) / (temp_c + 243.5))
    e_air = e_s_air * (df[h_col] / 100.0)
    rho_d_air = (press_hpa * 100.0) / (287.058 * temp_k)
    rho_v_air = (e_air * 100.0) / (461.495 * temp_k)
    df['air_pressure'] = (rho_d_air + rho_v_air) * 287.058 * temp_k / 100.0

    # 기존 P3. 풍속/풍향 물리 변환
    df['ws'] = np.sqrt(df[u_col]**2 + df[v_col]**2)
    df['wd'] = np.arctan2(-df[u_col], -df[v_col])
    df['ws_sin'] = df['ws'] * np.sin(df['wd'])
    df['ws_cos'] = df['ws'] * np.cos(df['wd'])
    
    # ----------------------------------------------------
    # P4. 격자 집계 (Grid-mean 전략) + 1위 신규 피처 일괄 집계
    # ----------------------------------------------------
    grouped = df.groupby('forecast_kst_dtm').agg({
        'ws': ['mean', 'max', 'std'],
        'ws_sin': 'mean',
        'ws_cos': 'mean',
        'air_density': 'mean',       
        'absolute_humidity': 'mean', 
        'air_pressure': 'mean'       
    })
    
    # 컬럼명 정리
    grouped.columns = [f"{prefix}_{c[0]}_{c[1]}" for c in grouped.columns]
    res_df = group_data = grouped.reset_index()
    
    # 시간 컬럼 datetime 변환 보강 (타입 충돌 차단)
    res_df['forecast_kst_dtm'] = pd.to_datetime(res_df['forecast_kst_dtm'])
    return res_df

print("Extracting NWP features...")
ldaps_features = process_nwp_features(ldaps_train, prefix='ldaps')
gfs_features = process_nwp_features(gfs_train, prefix='gfs')

# ==========================================
# P1 & P6: SCADA 전처리 및 Downtime 플래깅
# ==========================================
print("Processing SCADA glitches...")
# P1. Vestas 글리치 제거 (예시: wtg01)
# 물리상한 초과만 NaN 처리 (소량 음수는 유지)
vestas_cap = 780
scada_vestas['vestas_wtg01_power_kw10m'] = np.where(
    scada_vestas['vestas_wtg01_power_kw10m'] > vestas_cap * 1.3, 
    np.nan, 
    scada_vestas['vestas_wtg01_power_kw10m']
)

# P6. Downtime 구간 플래깅 (바람은 부는데 발전량은 0 이하인 고장 구간)
scada_vestas['is_downtime'] = np.where(
    (scada_vestas['vestas_wtg01_ws'] >= 5.0) & (scada_vestas['vestas_wtg01_power_kw10m'] <= 0), 
    1, 
    0
)
# 10분 단위를 1시간 단위로 평활화할 때 downtime 비율을 피처로 활용 가능

# ==========================================
# P7: Temporal Feature Engineering (시간 인코딩)
# ==========================================
def add_temporal_features(df, time_col):
    df['hour'] = df[time_col].dt.hour
    df['month'] = df[time_col].dt.month
    df['day_of_year'] = df[time_col].dt.dayofyear
    
    # Hour Cyclical Encoding
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24.0)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24.0)
    return df

# ==========================================
# 데이터 최종 조립 (Merge) 및 P2 보정
# ==========================================
print("Merging all features...")
# 정답 라벨 기준으로 조립 시작
final_df = train_labels.copy()

# NWP 예보 피처 병합
final_df = pd.merge(final_df, ldaps_features, left_on='kst_dtm', right_on='forecast_kst_dtm', how='left')
final_df = pd.merge(final_df, gfs_features, left_on='kst_dtm', right_on='forecast_kst_dtm', how='left')
final_df = add_temporal_features(final_df, 'kst_dtm')

# P2. Group 3 라벨 결측 처리 (Linear Imputation)
# 결측이 없는 구간으로 회귀 모델 학습 후 결측 구간 채우기
# === [수정] Group 1, 2 라벨도 결측치가 없는 깨끗한 행만 고르도록 필터링을 보강합니다 ===
# kpx_group_3도 있고, group_1, group_2도 모두 정상 수치인 행만 학습 데이터로 씁니다.
valid_g3 = final_df[
    final_df['kpx_group_3'].notnull() & 
    final_df['kpx_group_1'].notnull() & 
    final_df['kpx_group_2'].notnull()
]

# 예측해서 채워 넣어야 하는 구간은 기존과 동일하게 유지하되, 만약의 경우를 대비해 Group 1, 2 결측이 없는 구간으로 한정합니다.
missing_g3 = final_df[
    final_df['kpx_group_3'].isnull() & 
    final_df['kpx_group_1'].notnull() & 
    final_df['kpx_group_2'].notnull()
]

if len(missing_g3) > 0:
    print("Imputing Group 3 missing labels...")
    lr = LinearRegression()
    # 상관관계가 높은 Group 1, 2로 Group 3 예측
    lr.fit(valid_g3[['kpx_group_1', 'kpx_group_2']], valid_g3['kpx_group_3'])
    pred_g3 = lr.predict(missing_g3[['kpx_group_1', 'kpx_group_2']])
    final_df.loc[missing_g3.index, 'kpx_group_3'] = pred_g3

# ==========================================
# P10: Train / Validation Split
# ==========================================
# 시계열 temporal split 구조 적용
train_set = final_df[final_df['kst_dtm'].dt.year.isin([2022, 2023])]
valid_set = final_df[final_df['kst_dtm'].dt.year == 2024]

print(f"Preprocessing Done! Train shape: {train_set.shape}, Valid shape: {valid_set.shape}")

Cleaning NWP data...
Extracting NWP features...
Processing SCADA glitches...
Merging all features...
Imputing Group 3 missing labels...
Preprocessing Done! Train shape: (17519, 27), Valid shape: (8784, 27)


## 4. Feature 생성

LDAPS/GFS는 하나의 예측 시각에 여러 격자가 존재하므로, `forecast_kst_dtm`별로 수치형 기상변수의 평균값을 계산합니다.

추가로 월, 일, 시간, 요일, 주말 여부, 시간/월의 주기성을 나타내는 sin-cos feature를 생성합니다.


In [23]:
# 1. TEST 데이터도 똑같이 NWP 중복 제거 및 가용성 필터링 (P8)
print("Cleaning TEST NWP data...")
ldaps_test_clean = clean_nwp(ldaps_test)
gfs_test_clean = clean_nwp(gfs_test)

# 2. TEST 데이터도 똑같이 U/V -> 풍속/풍향 물리 변환 및 격자 집계 (P3, P4)
print("Extracting TEST NWP features...")
ldaps_test_features = process_nwp_features(ldaps_test_clean, prefix='ldaps')
gfs_test_features = process_nwp_features(gfs_test_clean, prefix='gfs')

# 3. TEST 데이터 최종 조립 (test_df 생성)
print("Merging TEST features...")
test_df = sample_submission[["forecast_id", "forecast_kst_dtm"]].copy()
test_df = pd.merge(test_df, ldaps_test_features, left_on='forecast_kst_dtm', right_on='forecast_kst_dtm', how='left')
test_df = pd.merge(test_df, gfs_test_features, left_on='forecast_kst_dtm', right_on='forecast_kst_dtm', how='left')

# 4. TEST 데이터에도 동일한 시간 파생 변수 인코딩 적용 (P7)
test_df = add_temporal_features(test_df, 'forecast_kst_dtm')

# 5. 모델 학습에 사용할 최종 피처(X)와 라벨(y) 확정 분리
drop_features = ["kst_dtm", "forecast_kst_dtm", "forecast_id", *TARGET_COLS]

# [TRAIN / VALID] - 3번 셀의 시계열 스플릿 결과물 활용 (P10)
X_train = train_set.drop(columns=drop_features, errors='ignore')
y_train_total = train_set[TARGET_COLS]

X_val = valid_set.drop(columns=drop_features, errors='ignore')
y_val_total = valid_set[TARGET_COLS]

# [TEST] - 7번 제출 파일 생성을 위해 모델에 들어갈 최종 테스트 피처
X_test = test_df.drop(columns=drop_features, errors='ignore')

print("\n--- 파이프라인 조립 완료 ---")
print("X_train (학습 피처):", X_train.shape)
print("X_val   (검증 피처):", X_val.shape)
print("X_test  (평가 피처):", X_test.shape)

Cleaning TEST NWP data...
Extracting TEST NWP features...
Merging TEST features...

--- 파이프라인 조립 완료 ---
X_train (학습 피처): (17519, 23)
X_val   (검증 피처): (8784, 23)
X_test  (평가 피처): (8760, 21)


## 5. 모델 학습

KPX 그룹별로 Label 제공 기간이 다르므로, 각 그룹의 Label이 존재하는 행만 사용해 RandomForest 모델을 따로 학습합니다.

RandomForest는 여러 결정트리를 앙상블하는 배깅(Bagging) 모델로, 각 트리를 부트스트랩 샘플과 랜덤 피처
subset으로 학습한 뒤 예측을 평균 냅니다.


In [24]:
# === 5번 셀(cell09) LightGBM 정밀 튜닝 버전 ===
import lightgbm as lgb
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

# 숫자형 컬럼 선별 및 결측치 채우기
X_train_numeric = X_train.select_dtypes(include=[np.number])
X_test_numeric = X_test.select_dtypes(include=[np.number])

X_train_imp = pd.DataFrame(imputer.fit_transform(X_train_numeric), columns=X_train_numeric.columns)
X_test_imp = pd.DataFrame(imputer.transform(X_test_numeric), columns=X_test_numeric.columns)

predictions = pd.DataFrame(index=sample_submission.index)
models = {}

print("Training Golden LightGBM Models...")
for target in TARGET_COLS:
    train_mask = train_set[target].notna() 
    y_train = train_set.loc[train_mask, target].values

    # P9. 정격 용량의 10% 미만 저출력 구간 가중치 부여
    threshold = CAPACITY_KWH[target] * 0.1
    sample_weights = np.where(y_train < threshold, 0.1, 1.0)

    # 🎯 Optuna가 찾아낸 가장 완벽한 황금 파라미터 주입
    model = lgb.LGBMRegressor(
        n_estimators=950,
        learning_rate=0.04895998685487635,
        max_depth=11,
        num_leaves=105,
        min_child_samples=18,
        subsample=0.7370710928872178,
        colsample_bytree=0.7127131803321877,
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )
    
    # 학습 진행
    model.fit(
        X_train_imp.loc[train_mask], 
        y_train, 
        sample_weight=sample_weights
    )
    models[target] = model

    # 예측 생성 및 클리핑
    pred = model.predict(X_test_imp)
    pred = np.clip(pred, 0, CAPACITY_KWH[target])
    predictions[target] = pred

    print(f"▶ {target} Golden LightGBM Train Done!")

Training Golden LightGBM Models...
▶ kpx_group_1 Golden LightGBM Train Done!
▶ kpx_group_2 Golden LightGBM Train Done!
▶ kpx_group_3 Golden LightGBM Train Done!


In [19]:
# === [Optuna 자동 튜닝 셀] LightGBM 황금 파라미터 찾기 ===
import optuna
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

print("🚀 Optuna 튜닝 시작 (30회 탐색)...")

# 1. 튜닝을 위해 학습 데이터의 일부를 검증용으로 분리 (1위 마스터 방식)
target_col = TARGET_COLS[0] # 대표 타겟 하나만 집중 공략
train_mask = train_set[target_col].notna()
X_opt = X_train_imp.loc[train_mask]
y_opt = train_set.loc[train_mask, target_col].values

X_tr, X_val, y_tr, y_val = train_test_split(X_opt, y_opt, test_size=0.2, random_state=42)

# 2. Optuna 목적 함수 정의
def objective(trial):
    # 6%~8% 경계선을 파고들기 위한 파라미터 탐색 공간 설정
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 300, 1000, step=50),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.05, log=True),
        'max_depth': trial.suggest_int('max_depth', 6, 15),
        'num_leaves': trial.suggest_int('num_leaves', 31, 127),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 40),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1
    }
    
    # 모델 학습
    model = lgb.LGBMRegressor(**params)
    model.fit(X_tr, y_tr)
    
    # 검증 데이터 예측 및 MAE 평가 (1위 마스터 방식)
    preds = model.predict(X_val)
    score = mean_absolute_error(y_val, preds)
    return score

# 3. 튜닝 실행
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=30) # 30번만 빠르게 탐색

print("\n🎯 [탐색 완료] 가장 완벽한 황금 파라미터:")
print(study.best_trial.params)

[I 2026-07-07 00:13:12,874] A new study created in memory with name: no-name-e04a47aa-0d97-42ca-8880-32797bcb8d7a


🚀 Optuna 튜닝 시작 (30회 탐색)...


[I 2026-07-07 00:13:16,195] Trial 0 finished with value: 1609.9659664328035 and parameters: {'n_estimators': 850, 'learning_rate': 0.029290363578080642, 'max_depth': 13, 'num_leaves': 100, 'min_child_samples': 22, 'subsample': 0.6661227072938256, 'colsample_bytree': 0.7497648151157753}. Best is trial 0 with value: 1609.9659664328035.
[I 2026-07-07 00:13:18,661] Trial 1 finished with value: 2087.108008998476 and parameters: {'n_estimators': 650, 'learning_rate': 0.005280158867344165, 'max_depth': 8, 'num_leaves': 99, 'min_child_samples': 20, 'subsample': 0.7669206853483388, 'colsample_bytree': 0.8584918778870216}. Best is trial 0 with value: 1609.9659664328035.
[I 2026-07-07 00:13:19,819] Trial 2 finished with value: 1779.450844775283 and parameters: {'n_estimators': 750, 'learning_rate': 0.032848225850508124, 'max_depth': 15, 'num_leaves': 33, 'min_child_samples': 39, 'subsample': 0.927019557616693, 'colsample_bytree': 0.9161831739555294}. Best is trial 0 with value: 1609.9659664328035


🎯 [탐색 완료] 가장 완벽한 황금 파라미터:
{'n_estimators': 950, 'learning_rate': 0.04895998685487635, 'max_depth': 11, 'num_leaves': 105, 'min_child_samples': 18, 'subsample': 0.7370710928872178, 'colsample_bytree': 0.7127131803321877}


## 6. 테스트 데이터 예측 생성

학습한 그룹별 모델로 평가 기간 전체의 발전량을 예측합니다. 예측값은 0 이상, 설비 용량 이하로 클리핑하여
물리적으로 불가능한 값을 방지합니다.


In [25]:
submission = sample_submission[["forecast_id", "forecast_kst_dtm"]].copy()
for target in TARGET_COLS:
    submission[target] = predictions[target]

submission["forecast_kst_dtm"] = pd.to_datetime(submission["forecast_kst_dtm"]).dt.strftime("%Y-%m-%d %H:%M:%S")

print(submission.head())
print(submission.shape)


     forecast_id     forecast_kst_dtm   kpx_group_1   kpx_group_2  \
0  forecast_0001  2025-01-01 01:00:00  16566.340008  19127.641577   
1  forecast_0002  2025-01-01 02:00:00  16194.415340  19295.729502   
2  forecast_0003  2025-01-01 03:00:00  17503.768189  19830.522798   
3  forecast_0004  2025-01-01 04:00:00  17262.607439  19822.438262   
4  forecast_0005  2025-01-01 05:00:00  17004.991388  19760.680749   

    kpx_group_3  
0  16284.678992  
1  15937.280215  
2  16367.788976  
3  15949.186728  
4  15995.068022  
(8760, 5)


## 7. 평가 산식

In [26]:
# === 6. 테스트 데이터 예측 생성 아래에 새로 만들 셀 ===

# 데이콘 공식 메트릭 함수 정의 (상단에 한 번만 선언해 줍니다)
def metric(answer_df, pred_df):
    group_nmae = []
    group_ficr = []

    for col in TARGET_COLS:
        actual = answer_df[col].to_numpy(dtype=float)
        forecast = pred_df[col].to_numpy(dtype=float)
        capacity = CAPACITY_KWH[col]

        valid = actual >= capacity * 0.10

        actual = actual[valid]
        forecast = forecast[valid]

        error_rate = np.abs(forecast - actual) / capacity
        group_nmae.append(np.mean(error_rate))

        unit_price = np.select(
            [error_rate <= 0.06, error_rate <= 0.08],
            [4.0, 3.0],
            default=0.0,
        )

        earned_settlement = np.sum(actual * unit_price)
        max_settlement = np.sum(actual * 4.0)

        group_ficr.append(earned_settlement / max_settlement)

    one_minus_nmae = 1 - np.mean(group_nmae)
    ficr = np.mean(group_ficr)

    total_score = 0.5 * one_minus_nmae + 0.5 * ficr

    return total_score, one_minus_nmae, ficr


# ----------------------------------------------------
# 2024년 검증 데이터(X_val)에 대한 로컬 점수 계산 시작
# ----------------------------------------------------
print("Calculating Local Validation Score...")

# 1. 실제 2024년의 정답 라벨 데이터 준비 (Y)
answer_df = valid_set[TARGET_COLS].copy()

# 2. 5번 셀에서 학습이 완료된 모델들로 2024년 검증 피처 예측값 생성 (Y_pred)
X_val_numeric = X_val.select_dtypes(include=[np.number])
X_val_imp = pd.DataFrame(imputer.transform(X_val_numeric), columns=X_val_numeric.columns)

pred_df = pd.DataFrame(index=valid_set.index)

for target in TARGET_COLS:
    # 저장해둔 각 그룹별 모델(models[target])을 꺼내와서 예측 및 클리핑 수행
    pred_val = models[target].predict(X_val_imp)
    pred_df[target] = np.clip(pred_val, 0, CAPACITY_KWH[target])

# 3. 점수 산출
total_score, one_minus_nmae, ficr = metric(answer_df, pred_df)

print("\n==========================================")
print("🎯 LOCAL VALIDATION SCORE REPORT (2024)")
print("==========================================")
print(f"▶ 최종 점수 (Total Score) : {total_score:.5f}")
print(f"▶ 1-NMAE                 : {one_minus_nmae:.5f}")
print(f"▶ 정산금획득률 (FICR)     : {ficr:.5f}")
print("==========================================")

Calculating Local Validation Score...

🎯 LOCAL VALIDATION SCORE REPORT (2024)
▶ 최종 점수 (Total Score) : 0.58265
▶ 1-NMAE                 : 0.85999
▶ 정산금획득률 (FICR)     : 0.30531


## 8. baseline_submit.csv 생성

`sample_submission.csv` 형식에 맞춰 최종 제출 파일을 생성합니다. 이 대회는 결과 CSV를 직접 제출하는
방식이므로, 이 파일을 그대로 제출하면 됩니다.


In [27]:
output_path = DATA_DIR / "baseline_submit.csv"
submission.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"Saved: {output_path}")


Saved: C:\Users\user\iCloudDrive\riversun\동국대학교\3학년 1학기\대외활동\DACON\데이터\open\baseline_submit.csv
